In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
from sklearn.preprocessing import LabelEncoder
import networkx as nx
import warnings
import time
from scipy.special import rel_entr
from tqdm import tqdm
import gc  # Garbage collection

# Bayesian Network libraries
import bnlearn as bn
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator
from pgmpy.inference import VariableElimination

warnings.filterwarnings('ignore')
np.random.seed(42)

# ============================================================================
# 1. DATA LOADING AND PREPROCESSING (HEART DISEASE)
# ============================================================================

def load_and_preprocess_heart(filepath):
    """
    Load and preprocess the heart disease dataset.
    
    Expected columns (Kaggle):
    age, sex, cp, trestbps, chol, fbs, restecg, thalach,
    exang, oldpeak, slope, ca, thal, target
    """
    print("Loading Heart Disease dataset...")
    df = pd.read_csv(filepath)
    
    print(f"Dataset shape: {df.shape}")
    print(f"\nColumn names:\n{df.columns.tolist()}")
    print(f"\nFirst few rows:\n{df.head()}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    
    if 'target' not in df.columns:
        raise ValueError("Expected 'target' column in the dataset for heart disease label.")
    
    print(f"\nTarget distribution (heart disease):\n{df['target'].value_counts()}")
    
    # Handle missing values (if any) by median for numeric
    df = df.fillna(df.median(numeric_only=True))
    
    return df


# ============================================================================
# 2. “DISCRETIZATION” / TYPE NORMALISATION FOR HEART DATASET
# ============================================================================

def discretize_heart_dataset(df, target_col='target'):
    """
    For the heart disease dataset, values are already integer-coded / categorical-like.
    We simply ensure all columns are integer (discrete states) and do NOT apply binning.
    """
    print("\n" + "="*70)
    print("DISCRETIZING / NORMALISING HEART DATASET (NO REAL BINNING)")
    print("="*70)
    
    df_disc = df.copy()
    
    # Ensure all columns numeric; cast floats to int where reasonable
    numeric_cols = df_disc.select_dtypes(include=[np.number]).columns.tolist()
    
    # Make sure target is not accidentally altered (but remain integer)
    if target_col not in numeric_cols:
        raise ValueError(f"Target column '{target_col}' must be numeric.")
    
    print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
    print("No KBinsDiscretizer is applied; values are treated as discrete states.")
    
    # Cast everything numeric to int (oldpeak may be float; we round)
    for col in numeric_cols:
        # Round before casting if float
        if pd.api.types.is_float_dtype(df_disc[col]):
            df_disc[col] = df_disc[col].round().astype(int)
        else:
            df_disc[col] = df_disc[col].astype(int)
    
    print(f"\nDiscretized/normalized dataset shape: {df_disc.shape}")
    print(f"All numeric columns cast to int. Unique dtypes: {df_disc.dtypes.unique()}")
    
    # No label_encoders needed, but return placeholder for consistency
    label_encoders = {}
    return df_disc, label_encoders


# ============================================================================
# 3. STRUCTURE LEARNING USING BNLEARN (Hill Climb + BIC)
# ============================================================================

def learn_structure_bnlearn(data, target='target', methodtype='hc', scoretype='bic', max_indegree=3):
    """
    Learn Bayesian Network structure using bnlearn library.
    methodtype: 'hc' (hill climb), 'ex' (exhaustive), 'cs' (constraint-based)
    scoretype: 'bic', 'k2', 'bdeu'
    """
    print("\n" + "="*70)
    print(f"STRUCTURE LEARNING (HEART): BNLEARN ({methodtype.upper()} + {scoretype.upper()})")
    print("="*70)
    
    start_time = time.time()
    
    print(f"Learning structure with method={methodtype}, scoring={scoretype}...")
    print(f"Max indegree (parents per node): {max_indegree}")
    
    try:
        # Learn structure using bnlearn with constraints
        model = bn.structure_learning.fit(
            data,
            methodtype=methodtype,
            scoretype=scoretype,
            max_indegree=max_indegree,
            verbose=0
        )
        
        edges = model['model_edges']
        
    except Exception as e:
        print(f"Error in structure learning: {e}")
        print("Using simplified structure with target as child of most correlated features...")
        
        # Fallback: simple structure
        columns = [col for col in data.columns if col != target]
        correlations = data.corr()[target].abs().sort_values(ascending=False)
        top_features = correlations[1:6].index.tolist()
        
        edges = [(feat, target) for feat in top_features]
        model = {'model': None, 'model_edges': edges}
    
    elapsed_time = time.time() - start_time
    
    print(f"\nStructure learning completed in {elapsed_time:.2f} seconds")
    print(f"Number of edges learned: {len(edges)}")
    print("\nLearned edges:")
    for edge in edges[:20]:
        print(f"  {edge[0]} -> {edge[1]}")
    if len(edges) > 20:
        print(f"  ... and {len(edges) - 20} more edges")
    
    return model, edges, elapsed_time


# ============================================================================
# 4. PARAMETER LEARNING USING PGMPY (Maximum Likelihood Estimation)
# ============================================================================

def learn_parameters_pgmpy(data, edges, target='target', use_bayesian=False):
    """
    Learn parameters (CPTs) using pgmpy's Maximum Likelihood Estimator
    """
    print("\n" + "="*70)
    print("PARAMETER LEARNING (HEART): PGMPY (Maximum Likelihood Estimation)")
    print("="*70)
    
    start_time = time.time()
    
    # Ensure edges is a list of tuples
    if not edges:
        print("No edges provided. Creating simple model using most correlated features...")
        correlations = data.corr()[target].abs().sort_values(ascending=False)
        top_features = correlations[1:4].index.tolist()
        edges = [(feat, target) for feat in top_features]
    
    model = BayesianNetwork(edges)
    
    print("Fitting model with Maximum Likelihood Estimation...")
    
    try:
        if use_bayesian:
            model.fit(data, estimator=BayesianEstimator,
                      prior_type='BDeu', equivalent_sample_size=10)
        else:
            model.fit(data, estimator=MaximumLikelihoodEstimator)
        
        if not model.check_model():
            print("WARNING: Model structure may have issues (check_model() failed).")
            
    except Exception as e:
        print(f"Error in parameter learning: {e}")
        print("Retrying with Bayesian Estimator...")
        model.fit(data, estimator=BayesianEstimator,
                  prior_type='BDeu', equivalent_sample_size=10)
    
    elapsed_time = time.time() - start_time
    
    print(f"\nParameter learning completed in {elapsed_time:.2f} seconds")
    print(f"Number of CPTs learned: {len(model.get_cpds())}")
    
    return model, elapsed_time


# ============================================================================
# 5. INFERENCE USING PGMPY (with batching)
# ============================================================================

def predict_proba_pgmpy(test_data, model, target='target', batch_size=100):
    """
    Predict probabilities using pgmpy's Variable Elimination inference.
    """
    predictions = []
    
    # Evidence variables: all nodes except target
    evidence_vars = [col for col in test_data.columns if col != target and col in model.nodes()]
    
    if not evidence_vars:
        print("WARNING: No evidence variables found in model.")
        return np.array([0.5] * len(test_data))
    
    print(f"\nPerforming inference on {len(test_data)} samples...")
    print(f"Evidence variables used: {len(evidence_vars)}")
    
    num_batches = (len(test_data) + batch_size - 1) // batch_size
    
    for batch_idx in tqdm(range(num_batches), desc="Inference Batches", ncols=100):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(test_data))
        batch_data = test_data.iloc[start_idx:end_idx]
        
        for _, row in batch_data.iterrows():
            try:
                inference = VariableElimination(model)
                
                evidence = {}
                for var in evidence_vars:
                    try:
                        evidence[var] = int(row[var])
                    except Exception:
                        pass
                
                if not evidence:
                    predictions.append(0.5)
                    continue
                
                result = inference.query(variables=[target],
                                         evidence=evidence,
                                         show_progress=False)
                
                if hasattr(result, 'values'):
                    if len(result.values) > 1:
                        # assuming state 1 = heart disease present
                        prob_pos = float(result.values[1])
                    else:
                        prob_pos = 0.5
                else:
                    prob_pos = 0.5
                
                prob_pos = np.clip(prob_pos, 0.0, 1.0)
                predictions.append(prob_pos)
            
            except Exception:
                predictions.append(0.5)
        
        gc.collect()
    
    return np.array(predictions)


def predict_pgmpy(test_data, model, target='target', threshold=0.5):
    """Predict class labels from predicted probabilities."""
    proba = predict_proba_pgmpy(test_data, model, target)
    return (proba >= threshold).astype(int)


# ============================================================================
# 6. EVALUATION METRICS
# ============================================================================

def calculate_kl_divergence(y_true, y_pred_proba):
    """Calculate average Kullback-Leibler Divergence."""
    try:
        y_true_proba = np.zeros((len(y_true), 2))
        y_true_proba[np.arange(len(y_true)), y_true] = 1
        
        y_pred_dist = np.column_stack([1 - y_pred_proba, y_pred_proba])
        
        epsilon = 1e-10
        y_true_proba = y_true_proba + epsilon
        y_pred_dist = y_pred_dist + epsilon
        
        y_true_proba /= y_true_proba.sum(axis=1, keepdims=True)
        y_pred_dist /= y_pred_dist.sum(axis=1, keepdims=True)
        
        kl = np.sum(rel_entr(y_true_proba, y_pred_dist))
        return kl / len(y_true)
    except Exception:
        return np.nan


def calculate_ece(y_true, y_pred_proba, n_bins=10):
    """Calculate Expected Calibration Error."""
    try:
        bins = np.linspace(0, 1, n_bins + 1)
        bin_indices = np.digitize(y_pred_proba, bins) - 1
        bin_indices = np.clip(bin_indices, 0, n_bins - 1)
        
        ece = 0
        for i in range(n_bins):
            mask = bin_indices == i
            if np.sum(mask) > 0:
                bin_accuracy = np.mean(y_true[mask])
                bin_confidence = np.mean(y_pred_proba[mask])
                bin_weight = np.sum(mask) / len(y_true)
                ece += bin_weight * np.abs(bin_accuracy - bin_confidence)
        
        return ece
    except Exception:
        return np.nan


def evaluate_model(y_true, y_pred, y_pred_proba):
    """Calculate all evaluation metrics."""
    metrics = {}
    
    try:
        metrics['accuracy'] = accuracy_score(y_true, y_pred)
    except Exception:
        metrics['accuracy'] = np.nan
    
    try:
        metrics['auc'] = roc_auc_score(y_true, y_pred_proba)
    except Exception:
        metrics['auc'] = np.nan
    
    try:
        metrics['brier_score'] = brier_score_loss(y_true, y_pred_proba)
    except Exception:
        metrics['brier_score'] = np.nan
    
    metrics['kl_divergence'] = calculate_kl_divergence(y_true, y_pred_proba)
    metrics['ece'] = calculate_ece(y_true, y_pred_proba)
    
    return metrics


# ============================================================================
# 7. CROSS-VALIDATION FOR HEART DATASET
# ============================================================================

def cross_validate_bn(data, target='target', k=5,
                      methodtype='hc', scoretype='bic',
                      max_indegree=3, sample_size=None):
    """
    Perform K-fold cross-validation for Bayesian Network on heart dataset.
    """
    print("\n" + "="*70)
    print(f"CROSS-VALIDATION (HEART, K={k})")
    print("="*70)
    
    if sample_size and len(data) > sample_size:
        print(f"\nSampling {sample_size} instances from {len(data)} for faster computation...")
        data = data.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    cv_results = {
        'accuracy': [],
        'auc': [],
        'brier_score': [],
        'kl_divergence': [],
        'ece': [],
        'train_time': [],
        'test_time': []
    }
    
    fold = 1
    final_model = None
    final_edges = None
    
    for train_idx, test_idx in kf.split(data):
        print(f"\n{'='*70}")
        print(f"FOLD {fold}/{k}")
        print(f"{'='*70}")
        
        train_data = data.iloc[train_idx].reset_index(drop=True)
        test_data = data.iloc[test_idx].reset_index(drop=True)
        
        print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")
        
        try:
            # Structure learning
            bn_model, edges, struct_time = learn_structure_bnlearn(
                train_data,
                target=target,
                methodtype=methodtype,
                scoretype=scoretype,
                max_indegree=max_indegree
            )
            
            # Parameter learning
            pgmpy_model, param_time = learn_parameters_pgmpy(
                train_data, edges, target
            )
            
            train_time = struct_time + param_time
            
            # Testing
            print(f"\nTesting on fold {fold}...")
            test_start = time.time()
            y_true = test_data[target].values
            y_pred_proba = predict_proba_pgmpy(test_data, pgmpy_model,
                                               target, batch_size=100)
            y_pred = (y_pred_proba >= 0.5).astype(int)
            test_time = time.time() - test_start
            
            # Evaluation
            print(f"\nCalculating metrics...")
            metrics = evaluate_model(y_true, y_pred, y_pred_proba)
            
            print(f"\n{'='*70}")
            print(f"FOLD {fold} RESULTS (HEART):")
            print(f"{'='*70}")
            print(f"  Accuracy:       {metrics['accuracy']:.4f}")
            print(f"  AUC:            {metrics['auc']:.4f}")
            print(f"  Brier Score:    {metrics['brier_score']:.4f}")
            print(f"  KL Divergence:  {metrics['kl_divergence']:.4f}")
            print(f"  ECE:            {metrics['ece']:.4f}")
            print(f"  Train Time:     {train_time:.2f}s")
            print(f"  Test Time:      {test_time:.2f}s")
            
            for key in cv_results.keys():
                cv_results[key].append(metrics.get(key, train_time if key == 'train_time' else test_time))
            
            cv_results['train_time'][-1] = train_time
            cv_results['test_time'][-1] = test_time
            
            final_model = bn_model
            final_edges = edges
        
        except Exception as e:
            print(f"\n*** ERROR IN FOLD {fold}: {e} ***")
            print("Skipping this fold...")
        
        gc.collect()
        fold += 1
    
    return cv_results, final_model, final_edges


# ============================================================================
# 8. VISUALISATION
# ============================================================================

def visualize_dag(edges, all_nodes, target='target', save_path='heart_bn_dag.png'):
    """Visualize the learned DAG structure for heart dataset."""
    print("\nVisualizing DAG for Heart Disease BN...")
    
    try:
        G = nx.DiGraph()
        G.add_nodes_from(all_nodes)
        G.add_edges_from(edges)
        
        plt.figure(figsize=(16, 12))
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
        
        node_colors = ['red' if node == target else 'lightblue' for node in G.nodes()]
        
        nx.draw(
            G, pos,
            with_labels=True,
            node_color=node_colors,
            node_size=3000,
            font_size=8,
            font_weight='bold',
            arrows=True,
            arrowsize=20,
            edge_color='gray',
            width=2,
            arrowstyle='->',
            connectionstyle='arc3,rad=0.1'
        )
        
        plt.title("Learned Bayesian Network Structure - Heart Disease", fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"DAG saved to {save_path}")
        plt.close()
    except Exception as e:
        print(f"Error visualizing DAG: {e}")


def plot_cv_results(cv_results, save_path='heart_cv_results.png'):
    """Plot cross-validation results for heart dataset."""
    print("\nPlotting cross-validation results (Heart Disease)...")
    
    try:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        fig.suptitle('Cross-Validation Results - Heart Disease BN', fontsize=16, fontweight='bold')
        
        metrics = ['accuracy', 'auc', 'brier_score', 'kl_divergence', 'ece', 'train_time']
        titles = ['Accuracy', 'AUC', 'Brier Score', 'KL Divergence', 'ECE', 'Train Time (s)']
        
        for idx, (metric, title) in enumerate(zip(metrics, titles)):
            ax = axes[idx // 3, idx % 3]
            values = [v for v in cv_results[metric] if not np.isnan(v)]
            
            if values:
                ax.bar(range(1, len(values) + 1), values, alpha=0.7)
                ax.axhline(np.mean(values), linestyle='--',
                           label=f'Mean: {np.mean(values):.4f}')
                ax.set_xlabel('Fold', fontweight='bold')
                ax.set_ylabel(title, fontweight='bold')
                ax.set_title(title, fontweight='bold')
                ax.legend()
                ax.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"CV results plot saved to {save_path}")
        plt.close()
    except Exception as e:
        print(f"Error plotting CV results: {e}")


def print_final_results(cv_results):
    """Print final summary statistics for heart dataset."""
    print("\n" + "="*70)
    print("FINAL RESULTS - HEART DISEASE BN (SUMMARY STATISTICS)")
    print("="*70)
    
    metrics = ['accuracy', 'auc', 'brier_score', 'kl_divergence', 'ece', 'train_time', 'test_time']
    titles = ['Accuracy', 'AUC', 'Brier Score', 'KL Divergence', 'ECE', 'Train Time (s)', 'Test Time (s)']
    
    print(f"\n{'Metric':<20} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
    print("-" * 70)
    
    for metric, title in zip(metrics, titles):
        values = [v for v in cv_results[metric] if not np.isnan(v)]
        if values:
            mean_val = np.mean(values)
            std_val = np.std(values)
            min_val = np.min(values)
            max_val = np.max(values)
            print(f"{title:<20} {mean_val:<12.4f} {std_val:<12.4f} {min_val:<12.4f} {max_val:<12.4f}")


# ============================================================================
# 9. MAIN EXECUTION FOR HEART DATASET
# ============================================================================

def main():
    """Main execution function for Heart Disease discrete BN."""
    print("="*70)
    print("DISCRETE BAYESIAN NETWORK - HEART DISEASE DIAGNOSIS")
    print("Using: bnlearn (Structure) + pgmpy (Parameters & Inference)")
    print("="*70)
    
    # 1. Load data
    filepath = '../dataset/heart.csv'   # <<< UPDATE THIS PATH TO YOUR HEART CSV
    df = load_and_preprocess_heart(filepath)
    
    # 2. “Discretize” / normalise (no binning, only int-casting)
    df_discretized, label_encoders = discretize_heart_dataset(df, target_col='target')
    
    # 3. Cross-validation
    sample_size = None  # data is small (1025), no need to subsample
    cv_results, final_model, final_edges = cross_validate_bn(
        df_discretized,
        target='target',
        k=5,
        methodtype='hc',   # hill climb
        scoretype='bic',   # BIC scoring
        max_indegree=3,
        sample_size=sample_size
    )
    
    # 4. Print summary
    print_final_results(cv_results)
    
    # 5. Visualisations
    if final_edges:
        print("\n" + "="*70)
        print("GENERATING VISUALISATIONS (HEART DISEASE)")
        print("="*70)
        
        visualize_dag(final_edges, df_discretized.columns.tolist(),
                      target='target', save_path='heart_bn_dag.png')
        plot_cv_results(cv_results, save_path='heart_cv_results.png')
    
    print("\n" + "="*70)
    print("HEART DISEASE BN ANALYSIS COMPLETE")
    print("="*70)
    
    return cv_results, final_model, final_edges, df_discretized


if __name__ == "__main__":
    cv_results, bn_model, edges, discretized_data = main()
